# 🧬 Phase 1: Biomedical KG — Data Download & RAPIDS GPU Preprocessing

**Project:** BioKG Disease Link Predictor  
**Dataset:** DRKG (Drug Repurposing Knowledge Graph) — 5.8M biological triples  
**Tech:** NVIDIA RAPIDS cuDF/cuGraph + DRKG exploration  

---

## 🎓 What You'll Learn in This Notebook

1. **What a Knowledge Graph is** — entities, relations, triples
2. **NVIDIA RAPIDS cuDF** — GPU-accelerated DataFrames (10-100x faster than pandas)
3. **cuGraph** — GPU graph analytics algorithms (PageRank, degree statistics)
4. **DRKG structure** — the real biomedical dataset used in COVID-19 drug repurposing

## ⚡ Before Starting
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Click **Connect** (top right)
3. Run all cells in order (Shift+Enter)

## Step 0: Check GPU

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
if not torch.cuda.is_available():
    raise RuntimeError('❌ No GPU found! Go to Runtime → Change runtime type → T4 GPU and restart.')

print(f'\n✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'   CUDA (via torch): {torch.version.cuda}')

## Step 1: Install NVIDIA RAPIDS

**🎓 What is RAPIDS?**

NVIDIA RAPIDS is a suite of GPU-accelerated data science libraries:
- **cuDF**: GPU DataFrames (pandas API on GPU)
- **cuGraph**: GPU Graph Analytics (NetworkX API on GPU)

This cell will take ~3-5 minutes. ☕

In [ ]:
import sys
import torch
import subprocess

cuda_version = torch.version.cuda
major = int(cuda_version.split('.')[0]) if cuda_version else 12
rapids_suffix = 'cu11' if major == 11 else 'cu12'

install_cmd = f'pip install cudf-{rapids_suffix} cugraph-{rapids_suffix} --extra-index-url=https://pypi.nvidia.com -q'
print(f'📦 Installing NVIDIA RAPIDS {rapids_suffix}... ☕')

result = subprocess.run(install_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print('⚠️ STDERR:', result.stderr[-500:])
else:
    print('\n✅ RAPIDS installed successfully!')

In [ ]:
!pip install loguru rich tqdm requests -q

import cudf
import cugraph
print(f'✅ cuDF: {cudf.__version__} | cuGraph: {cugraph.__version__}')

## Step 2: Download & Extract DRKG

> ⚠️ **Fix Note:** We need to ignore hidden macOS files (those starting with `._`) during extraction and selection.

In [ ]:
import requests
from pathlib import Path
from tqdm.notebook import tqdm
import tarfile

DATA_DIR = Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

def download_file(url, dest_path, desc='Downloading'):
    if dest_path.exists():
        print(f'⏭️  {dest_path.name} already exists.')
        return
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total = int(response.headers.get('content-length', 0))
    with open(dest_path, 'wb') as f, tqdm(desc=desc, total=total, unit='iB', unit_scale=True) as pb:
        for chunk in response.iter_content(8192):
            pb.update(f.write(chunk))
    print(f'✅ Saved: {dest_path.name}')

print('📥 Downloading DRKG dataset...')
download_file('https://dgl-data.s3-accelerate.amazonaws.com/dataset/DRKG/drkg.tar.gz', DATA_DIR / 'drkg.tar.gz')

print('\n📦 Extracting (ignoring hidden files)...')
with tarfile.open(DATA_DIR / 'drkg.tar.gz', 'r:gz') as tar:
    # Only extract members that don't start with '._'
    members = [m for m in tar.getmembers() if not Path(m.name).name.startswith('._')]
    tar.extractall(path=DATA_DIR, members=members)

# Look for the REAL drkg.tsv
tsv_files = [f for f in DATA_DIR.rglob('*.tsv') if not f.name.startswith('._')]
DRKG_TSV = next((f for f in tsv_files if f.name == 'drkg.tsv'), tsv_files[0] if tsv_files else None)

if not DRKG_TSV or not DRKG_TSV.exists():
    raise FileNotFoundError("Could not find the main drkg.tsv file! Check extraction.")

print(f'\n✅ Loaded main file: {DRKG_TSV}')
print(f'   Size: {DRKG_TSV.stat().st_size / 1024**2:.1f} MB')

## Step 3: GPU Benchmark (cuDF vs pandas)

> 📝 **MLOps Pro-Tip:** We use `encoding='latin-1'` to handle non-standard characters in chemical names.

In [ ]:
import time
import cudf
import pandas as pd

print('⚡ BENCHMARK: GPU vs CPU')

print('\n🐢 Loading with pandas (CPU)...')
t0 = time.time()
df_pandas = pd.read_csv(str(DRKG_TSV), sep='\t', header=None, names=['head', 'relation', 'tail'], encoding='latin-1')
cpu_time = time.time() - t0
print(f'   Time: {cpu_time:.2f}s | Rows: {len(df_pandas):,}')

print('\n⚡ Loading with cuDF (GPU)...')
t0 = time.time()
df = cudf.read_csv(str(DRKG_TSV), sep='\t', header=None, names=['head', 'relation', 'tail'], encoding='latin-1')
gpu_time = time.time() - t0
print(f'   Time: {gpu_time:.2f}s | Rows: {len(df):,}')

RAPIDS_SPEEDUP = cpu_time / max(gpu_time, 0.001)
print(f'\n🚀 cuDF is {RAPIDS_SPEEDUP:.1f}x faster than pandas!')

## Step 4: Explore & Convert to Parquet

**🎓 Why Parquet?**  
Parquet is a binary columnar format. It's much smaller than TSV and significantly faster for GPUs to read. In real MLOps pipelines, we **never** train on raw TSV; we convert to Parquet first.

In [ ]:
print('📦 Converting to Parquet for MLOps optimization...')
PARQUET_PATH = Path('data/processed/drkg.parquet')
PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_parquet(PARQUET_PATH)
print(f'✅ Saved: {PARQUET_PATH} ({PARQUET_PATH.stat().st_size / 1024**2:.1f} MB)')

print('\n📊 Quick Stat: Entity Type Distribution (GPU)')
head_types = df['head'].str.split('::').list.get(0).value_counts().to_pandas()
print(head_types.head(10))

## Step 5: cuGraph PageRank

Let's find the 'hub' nodes of the biomedical world.

In [ ]:
import cugraph

all_entities = cudf.concat([df['head'], df['tail']]).unique().reset_index(drop=True)
entity_to_id = cudf.Series(range(len(all_entities)), index=all_entities)
edges = cudf.DataFrame({'src': df['head'].map(entity_to_id), 'dst': df['tail'].map(entity_to_id)}).dropna().astype('int32')

G = cugraph.Graph(directed=True)
G.from_cudf_edgelist(edges, source='src', destination='dst')

print('⚡ Computing PageRank on GPU...')
pr = cugraph.pagerank(G)
print('✅ Done!')

In [ ]:
id_to_entity = {int(v): k for k, v in entity_to_id.to_pandas().items()}
top_pr = pr.sort_values('pagerank', ascending=False).head(10).to_pandas()
top_pr['entity'] = top_pr['vertex'].map(id_to_entity)
print(top_pr[['entity', 'pagerank']])

## Step 6: Split & Save To Drive

In [ ]:
import json
from google.colab import drive
import shutil

PROCESSED_DIR = Path('data/processed')
unique_entities = cudf.concat([df['head'], df['tail']]).unique().to_pandas()
entity2id = {e: i for i, e in enumerate(sorted(unique_entities))}
with open(PROCESSED_DIR / 'entity2id.json', 'w') as f:
    json.dump(entity2id, f)

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/BioKG_Project/processed')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
for f in PROCESSED_DIR.iterdir():
    shutil.copy2(f, DRIVE_DIR / f.name)
print(f'🎉 Everything saved to Drive: {DRIVE_DIR}')